# 08 · Concept Chatbot Inference
Interactive chatbot demo: upload a file, process it through the concept pipeline, generate a response.

**Note:** Replace the untrained model with a fine-tuned checkpoint for real responses.

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
from transformers import GPT2Tokenizer
from src.model import ConceptLM, ConceptLMConfig
from src.file_parser import FileParser, segments_to_text

## 1. Initialize model and tokenizer

In [ ]:
cfg = ConceptLMConfig(
    d_token=256, n_token_layers=4, n_token_heads=8,
    d_concept=512, n_concept_layers=8, n_concept_heads=8,
    d_scan=128, vocab_size=50257, max_seq_len=2048,
    target_ratio=4,
)
model = ConceptLM(cfg)
# Production: model.load_state_dict(torch.load('checkpoints/best.pt')['model_state'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = model.to(device).eval()

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
parser_f  = FileParser()

total = sum(p.numel() for p in model.parameters())
print(f"Model: {total:,} parameters  |  device: {device}")

## 2. File ingestion pipeline

In [ ]:
def encode_file(filepath, max_tokens=1024):
    segs   = parser_f.parse(filepath)
    text   = segments_to_text(segs, max_chars=max_tokens * 4)
    ids    = tokenizer.encode(text, max_length=max_tokens, truncation=True)
    ids_t  = torch.tensor([ids], device=device)
    with torch.no_grad():
        H = model.encoder(ids_t)
    return H, ids_t

# Write test file
test_content = """# DLCM Architecture Notes

The encoder E processes raw tokens x to produce hidden states H.
The boundary detector computes boundary probabilities using cosine dissimilarity.
Tokens are pooled into concept vectors C via mean pooling within each segment.
The concept transformer M reasons over the compressed sequence C.
The decoder reconstructs token predictions via causal cross-attention.
"""
with open('/tmp/test_notes.md', 'w') as f:
    f.write(test_content)

H_file, ids_file = encode_file('/tmp/test_notes.md')
print(f"File encoded: {ids_file.shape[1]} tokens -> H shape: {H_file.shape}")

## 3. Concept segmentation inspection

In [ ]:
with torch.no_grad():
    b_inf, p_inf = model.boundary(H_file, training=False)
    C_list, seg_maps = model.pooler(H_file, b_inf)
    C_inf = C_list[0]
    Z_inf = model.concept_transformer(C_inf)

n_tokens   = ids_file.shape[1]
n_concepts = C_inf.shape[0]
print(f"Tokens: {n_tokens}  ->  Concepts: {n_concepts}")
print(f"Actual ratio: {n_tokens / n_concepts:.2f} tokens/concept  (target={cfg.target_ratio})")
print()
segs_map_dict = {}
for pos, con in enumerate(seg_maps[0].tolist()):
    segs_map_dict.setdefault(con, []).append(pos)

print("Concept -> token mapping (first 5):")
for ci, positions in list(segs_map_dict.items())[:5]:
    toks_text = tokenizer.decode(ids_file[0, positions[0]:positions[-1]+1].tolist())
    print(f"  C{ci}: {toks_text!r:.65s}")

## 4. Chat function

In [ ]:
def chat(user_message, file_path=None, max_new_tokens=64, temperature=0.8):
    parts = []
    if file_path:
        segs = parser_f.parse(file_path)
        parts.append("<file>\n" + segments_to_text(segs, max_chars=2000) + "\n</file>")
    parts.append("<user>\n" + user_message + "\n</user>")
    parts.append("<assistant>\n")
    full = "\n".join(parts)
    ids  = tokenizer.encode(full, max_length=cfg.max_seq_len - max_new_tokens, truncation=True)
    ids_t = torch.tensor([ids], device=device)
    with torch.no_grad():
        gen = model.generate(ids_t, max_new_tokens=max_new_tokens, temperature=temperature)
    new_toks = gen[0, len(ids):].tolist()
    return tokenizer.decode(new_toks, skip_special_tokens=True).strip()

resp = chat(
    user_message="What are the five stages of the DLCM architecture?",
    file_path='/tmp/test_notes.md', max_new_tokens=80)
print("Response (untrained model -- demonstrates pipeline structure):")
print(resp)

## 5. Attention complexity: concept LM vs. standard Transformer

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

lengths = [64, 128, 256, 512, 1024, 2048]
std_flops  = [l**2 for l in lengths]
con_flops  = [(l // cfg.target_ratio)**2 for l in lengths]

plt.figure(figsize=(8, 4))
plt.plot(lengths, std_flops,  '-o', color='#C00000', label='Standard LLM O(L^2)')
plt.plot(lengths, con_flops,  '-o', color='#2E75B6', label=f'Concept LM O((L/R)^2), R={cfg.target_ratio}')
plt.xlabel('Sequence length L'); plt.ylabel('Attention FLOPs (relative)')
plt.title('Concept-Level Attention Complexity vs. Standard Transformer')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('../data/samples/flop_comparison.png', dpi=150); plt.show()
print(f"At L=2048: concept LM uses {(1/cfg.target_ratio)**2 * 100:.1f}% of standard attention FLOPs")
print(f"R^2 = {cfg.target_ratio**2}x quadratic savings from concept-level attention")

## Next Steps

1. **Train:** Run `src/train.py` on OpenWebText or The Pile with joint NTP + boundary aux loss
2. **Fine-tune:** Apply COCONUT-style curriculum (NB07) to transfer to instruction following
3. **Evaluate:** Concept-level AutoBLEU, standard perplexity, zero-shot benchmarks vs. matched-FLOP standard LLM
4. **Scale:** Use DLCM compression-aware scaling law to determine optimal R and P for your compute budget before full training runs

**Central claim to validate:** At matched inference FLOPs, a concept-level LM outperforms a standard token LLM on reasoning-heavy benchmarks because it allocates compute proportional to information density.